## corner study
for the sake of time, we will not be looking at all corners over all the laps, instead we will try to take some corners that cover different types of corners for this track:
* C1 bc it's a wide fast
* [C2, C3, C4, C5] together bc they are esses
* C20 bc it's 90 degrees
* C24 and C27 bc hairpins
* C25 and C26 bc they're a snail corner

We will take the esses and the snail as one group and one window instead of two separate corners because it makes more sense.

and in the comparison, we will not take all these corners from all the laps, instead we will take:
* the corner of the reference lap (fastest lap)
* the fastest lap on that corner/group if it beats the reference one
* the slowest in stint 1
* the slowest in stint 2


In [1]:
from src.segment import load_segmented
from src.loader import load_centerline
from src.config import CORNER_GROUPS
from src.analysis import comparable_laps, reference_lap, corner_entries, group_times, pick_laps

In [2]:
data = load_segmented()
grid, laps, zones, apexes, gaps = data["grid"], data["laps"], data["zones"], data["apexes"], data["gaps"]
cl = load_centerline(); length = cl.attrs["length"]
laps_used = comparable_laps(laps)["lap"].tolist()
ref = reference_lap(laps)

entries = corner_entries(grid, apexes, zones, laps_used, length)
gt = group_times(grid, apexes, entries, zones, gaps, laps_used, CORNER_GROUPS, length)
picks = pick_laps(gt, ref)

In [3]:
picks

,group,role,lap,time,delta
0,C1,ref,10,4.083430,0.000000
1,C1,fastest,10,4.083430,0.000000
2,C1,slowest stint 1,3,4.930340,0.846910
3,C1,slowest stint 2,15,5.651795,1.568365
4,esses,ref,10,5.894141,0.000000
5,esses,fastest,22,5.686142,-0.207999
6,esses,slowest stint 1,3,6.541059,0.646918
7,esses,slowest stint 2,15,6.651147,0.757006
8,C20,ref,10,2.532906,0.000000
9,C20,fastest,21,2.355891,-0.177015


In [4]:
import numpy as np
import plotly.graph_objects as go
from plotly.colors import qualitative
from plotly.subplots import make_subplots
from src.segment import U_COL, THROTTLE_COL, BRAKE_COL, STEER_COL

CHANNELS = [(U_COL, "speed (m/s)"), (THROTTLE_COL, "throttle (0–1)"),
            (BRAKE_COL, "brake front (bar)"), (STEER_COL, "steering (deg, + = left)")]
COAST = "coasting" # phase label for both pedals off
main_apex = apexes[apexes["is_main"]].set_index("corner")["s"]


def plot_group(name: str) -> go.Figure:
    p = picks[picks["group"] == name]
    roles = p.groupby("lap", sort=False)["role"].agg(" + ".join)    # ref + fastest merged if same lap
    w = gt[gt["group"] == name].iloc[0]
    s0, s1 = float(w["s_start"]), float(w["s_end"])
    span = np.mod(s1 - s0, length)
    corners = CORNER_GROUPS[name]

    fig = make_subplots(rows=len(CHANNELS), cols=1, shared_xaxes=True, vertical_spacing=0.03)
    for k, (lap, role) in enumerate(roles.items()):
        lg = grid[grid["lap"] == lap].copy()
        lg["d"] = np.mod(lg["s"] - s0, length)                       # metres from window start
        lg = lg[lg["d"] <= span].sort_values("d").reset_index(drop=True)
        color = qualitative.Plotly[k % len(qualitative.Plotly)]
        label = f"lap {lap} ({role})"

        coast = (lg["phase"] == COAST).to_numpy()
        coast_ext = coast | np.roll(coast, 1) | np.roll(coast, -1)   # overlap one point so lines join
        lap_entries = entries[(entries["lap"] == lap) & (entries["corner"].isin(corners))]

        for i, (col, ylabel) in enumerate(CHANNELS, start=1):
            y = lg[col].to_numpy()
            fig.add_trace(go.Scatter(x=lg["d"], y=np.where(coast, np.nan, y), mode="lines",
                                     line=dict(color=color), name=label, legendgroup=str(lap),
                                     showlegend=(i == 1)), row=i, col=1)
            fig.add_trace(go.Scatter(x=lg["d"], y=np.where(coast_ext, y, np.nan), mode="lines",
                                     line=dict(color=color, dash="dash"), name=f"{label} coasting",
                                     legendgroup=str(lap), showlegend=False), row=i, col=1)

            for _, e in lap_entries.iterrows():                      # braking point / corner entry
                xe = float(np.mod(e["s_entry"] - s0, length))
                fig.add_trace(go.Scatter(x=[xe], y=[np.interp(xe, lg["d"], y)], mode="markers",
                                         marker=dict(color=color, size=11, symbol="triangle-down",
                                                     line=dict(color="black", width=1)),
                                         name=label, legendgroup=str(lap), showlegend=False,
                                         hovertext=f"lap {lap}: C{int(e['corner'])} entry ({e['entry_source']})",
                                         hoverinfo="text"), row=i, col=1)
            fig.update_yaxes(title_text=ylabel, row=i, col=1)

        u_exit = float(lg[U_COL].iloc[-1])                           # exit speed at window end
        fig.add_trace(go.Scatter(x=[float(lg["d"].iloc[-1])], y=[u_exit], mode="markers",
                                 marker=dict(color=color, size=12, symbol="diamond",
                                             line=dict(color="black", width=1)),
                                 name=label, legendgroup=str(lap), showlegend=False,
                                 hovertext=f"lap {lap}: exit {u_exit:.2f} m/s", hoverinfo="text"),
                      row=1, col=1)

    for c in corners:                                                # main apexes of the group
        x = float(np.mod(main_apex[c] - s0, length))
        fig.add_vline(x=x, line_dash="dot", line_color="grey")
        fig.add_annotation(x=x, y=1.0, yref="paper", text=f"C{c} apex", showarrow=False, yanchor="bottom")

    # legend keys for the symbols
    for sym, txt in [("triangle-down", "entry / braking point"), ("diamond", "exit speed")]:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=txt,
                                 marker=dict(color="grey", size=11, symbol=sym)), row=1, col=1)
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name="coasting",
                             line=dict(color="grey", dash="dash")), row=1, col=1)

    fig.update_xaxes(title_text="metres from window start", row=len(CHANNELS), col=1)
    fig.update_layout(title=f"{name}: picked laps", height=1050, width=1300,
                      legend=dict(orientation="h", y=1.07))
    return fig


In [5]:
# group C1, wide 
plot_group("C1").show()

Comparison of laps 3 and 15 with lap 10:
* Lap 15: there doesn't seem to be any braking point; the driver instead opted to lift at around 16.5 m. The driver was already entering the corner slowly, at 8.7 m/s and rising, and lifted when he reached around 11 m/s. After the apex the driver presses back on the throttle very little, and the speed picks up very gradually until he exits the corner at 13.85 m/s.
* Lap 3 on the other hand, brakes, and does it much earlier, at around 2.7 m from the start of the window. We notice that the driver coasts before hitting the brakes, so most likely they started lifting, **coasted for 1.5 m** and then hit the brakes, all this at 12.7 m/s. The transition between lifting and braking was not smooth: we expect the driver to lift off the throttle and press the brakes gradually, whereas here it looks more like they lifted off completely and then started braking. The braking pattern does not seem smooth at all either. We usually expect the trace to go up and then back down once, but instead it went up, then down, then up again and then down, so it seems like the driver was hesitant. The driver then lifts off the brakes at 12.7 m from the window start, at 11 m/s. The **coasting lasts about 10 m**, then the driver touches the throttle a tiny bit right before the apex and starts actually accelerating, with an upward trend, at 26.1 m (after the apex). The max throttle reaches 0.61, at a speed of 9.8 m/s, which then increases gradually, and he exits at 19.33 m/s.
* Lap 10: the driver enters at 15.2 m/s with the throttle at 0.73. He then starts lifting off the throttle and braking at the same time, gradually and smoothly, and starts to turn the car as well. The max braking reaches 9.5 bar and the throttle reaches 0. The braking lasts 13 m, with **1 m of coasting between ~16.4 and ~17.4 m**. The minimum speed is 12.2 m/s and is reached before the apex, and the speed is maintained through the corner's apex. The driver starts pressing the throttle gradually before the apex, at 18.6 m, and it goes up to 0.6 at 30 m. Then the driver lifts a little and goes back on the throttle at 37.8 m until it reaches 0.78, and it stays fairly steady with the speed rising until he exits at 21.59 m/s. The hesitation here on the throttle might have cost him some time, though.

* Additional notes:
  - A lot of the loss comes from C27, before C1 starts.

In [6]:
from src.analysis import corner_metrics
import pandas as pd
cm = corner_metrics(grid, entries, apexes, zones, laps_used, length)

# positions are metres from apex, 0 before apex and + after apex
SHOW = ["entry_from_apex", "peak_brake_bar", "brake_total_time", "coasting_time", "coasting_dist",
        "min_speed", "min_speed_from_apex", "pickup_from_apex", "max_exit_throttle", "exit_speed",
        "steer_corrections", "exit_slip_pct", "corner_time"]


def group_metrics(name: str) -> pd.DataFrame:
    p = picks[picks["group"] == name].drop_duplicates("lap")
    lap_list = p["lap"].tolist()
    m = cm[cm["lap"].isin(lap_list) & cm["corner"].isin(CORNER_GROUPS[name])]
    out = m.set_index(["corner", "lap"])[SHOW].T 

    s0 = float(gt.loc[gt["group"] == name, "s_start"].iloc[0])
    for lap in lap_list:   
        lg = grid[grid["lap"] == lap].sort_values("s")
        u0 = float(np.interp(s0, lg["s"], lg[U_COL]))
        r = p[p["lap"] == lap].iloc[0]
        for c in CORNER_GROUPS[name]:
            out.loc["speed_at_window_start", (c, lap)] = u0
            out.loc["group_time", (c, lap)] = r["time"]
            out.loc["group_delta_vs_ref", (c, lap)] = r["delta"]

    for c in CORNER_GROUPS[name]:
        print(f"C{c} apex at {np.mod(main_apex[c] - s0, length):.1f} m from window start")
    return out.round(2)

remember that from notebook 8:
**notice that the metric pickup_from_apex has NaN values, this could be one of three reasons, either the throttle was back on AFTER the corner zone end, the throttle was on the whole time so it never went off, or the throttle was never on anywhere in teh window.**

In [7]:
group_metrics("C1")

C1 apex at 24.7 m from window start


corner                     1              
lap                       3      10     15
entry_from_apex       -22.00 -21.40  -7.90
peak_brake_bar          3.73   9.55   0.00
brake_total_time        0.81   0.85   0.00
coasting_time           0.92   0.04   0.00
coasting_dist          10.50   0.60   0.00
min_speed               9.85  12.20  10.83
min_speed_from_apex     2.00   0.20  -5.80
pickup_from_apex       -2.20  -6.70    NaN
max_exit_throttle       0.62   0.78   0.23
exit_speed             19.33  21.59  13.85
steer_corrections       2.00   4.00   1.00
exit_slip_pct           1.82  19.79  -1.01
corner_time             4.73   3.87   4.00
speed_at_window_start  12.72  15.20   8.72
group_time              4.93   4.08   5.65
group_delta_vs_ref      0.85   0.00   1.57

in the case of lap 15, the NaN in pickup_from_apex happened is because the throttle never went off.

* **Metrics:**
  - The biggest difference between lap 3 and lap 10 is the coasting: lap 3 coasts for 0.92 s over 10.5 m, while lap 10 coasts for only 0.04 s over 0.6 m.
  - Lap 3 actually brakes for about as long as lap 10 (0.81 s vs 0.85 s), but with much less pressure (3.7 bar vs 9.5 bar). So the driver is not braking less, he is braking lighter and for longer, which matches the hesitation we see in the braking trace.
  - On lap 3 the minimum speed is reached 2 m after the apex, while on lap 10 it is reached right at the apex (0.2 m).
  - The throttle lift on lap 10 after the apex is most likely not hesitation: the rear wheels were slipping 19.8% at the exit and the driver made 4 steering corrections, against about 2% and 2 corrections on lap 3. So the rear was most likely sliding, and the driver lifted and corrected the steering to catch it. A smoother throttle pickup could make lap 10 even faster here.
  - The minimum speed is not the whole story: lap 15 has a higher minimum speed than lap 3 (10.8 m/s vs 9.9 m/s) but is still 0.7 s slower. What costs lap 15 the most is the slow entry (8.7 m/s) and the low exit speed (13.9 m/s), with the throttle never going above 0.23.
  - The time loss is measured on the same window for all laps: lap 3 loses 0.85 s and lap 15 loses 1.57 s compared to lap 10. (The corner time from each lap's own entry can't be compared here, because lap 15's entry is only 7.9 m before the apex, which makes its corner time look almost as good as lap 10's.)
  - Both lap 3 and lap 15 arrive at C1 slower than lap 10 (12.7 m/s and 8.7 m/s vs 15.2 m/s), and that carries into the corner. The time lost in C27 itself is not counted here; we will see how much it costs when we look at C27.

In [8]:
#  the esses
plot_group("esses").show()

Comparison of laps 22, 3 and 15 with lap 10 in the esses:
* Lap 10 (ref): the driver enters the fastest of all, at 22.85 m/s with the throttle at 0.57, and lifts off the throttle and starts braking at the same time at around 2 m. The braking is very hard, it goes up to 16 bar at around 11 m and stays between 12 and 15 bar until around 34 m, then the driver releases the brakes right before the C2 apex. The speed drops to around 9 m/s at the C2 apex. The driver then goes back on the throttle at the C2 apex up to 0.2, but lifts off again at around 43 m and brakes a second time for C3 (around 4.8 bar) before going back on the throttle at around 46 m. So between C2 and C3 the driver goes throttle, brake, throttle, which looks like he slowed down too much for C2 and then had to correct for C3. The minimum speed is around 8.1 m/s between C2 and C3. Between C3 and C4 the throttle drops again to around 0.1 while the steering is at full lock, then the driver gets back on the throttle from C4 up to 0.65 before C5 and exits the fastest of all, at 15.5 m/s.
* Lap 22 (fastest): the driver enters slower than lap 10, at 20.3 m/s, and is already coasting at the start of the window. He brakes at around 2.5 m but the braking builds up gradually and is much lighter than lap 10, around 9.5 bar at its peak and around 7 to 8.5 bar until he releases at the C2 apex. Even though he brakes less, he carries more speed into C2, around 10.7 m/s at the apex against 9 m/s for lap 10. He then coasts through the change of direction between C2 and C3 and does not brake at all for C3, and he also turns earlier and more than the others (the steering goes down to around −80deg). He stays the fastest through the middle of the esses with the speed between 9.3 and 10.3 m/s. Between C3 and C4 the throttle also drops to around 0.12 with a short coast before C4, and then goes up to 0.48 and stays steady until the exit. He exits slower than lap 10, at around 13.9 m/s, so his time is gained in the middle of the esses, not at the exit.
* Lap 3: the driver enters at 20.8 m/s with the throttle at 0.43, and starts braking at around 1.5 m. The braking goes up to around 11 bar and stays at around 10 bar until 33 m, but then the driver does not release before the apex like the others, the brake pressure goes down slowly and he is still braking at 5 to 6 bar after the C2 apex until around 41 m. Because of that he is the slowest through the middle, around 7.5 to 8 m/s between C2 and C3. He also holds the steering for C2 longer and changes direction later than the others, with some coasting during the change. He gets on the throttle after C3 up to 0.3, then lifts off completely and coasts for a few metres before C4, then gets back on the throttle gradually up to 0.5 and exits at around 13.6 m/s.
* Lap 15: the driver enters very slowly, at 14.2 m/s, and is coasting from the start of the window. He coasts for around 24 m and only brakes lightly at around 27 m, with a peak of around 3.7 bar. Since he entered so slowly, by the time the other drivers finish braking he is at the same speed as them, around 10.6 m/s at the C2 apex. He then brakes again for C3 at around 4 bar, for longer than lap 10, and gets on the throttle after C3 but only up to 0.22. He changes direction later than the others between C3 and C4, then gets back on the throttle up to around 0.4 but the throttle goes down again towards the exit, and he exits the slowest of all at around 12.6 m/s.

* **Additional notes:**
  - The fastest lap through the esses is not the one that brakes the hardest. Lap 22 brakes lighter, carries more speed into C2 and flows through C2 and C3 without braking again, while lap 10 brakes very hard, slows down too much and has to brake again for C3.
  - Lap 10 has the best exit, so the best way through the esses would be the middle of lap 22 with the exit of lap 10.
  - All the laps lift or drop the throttle between C3 and C4, so it looks like that part of the esses is hard to drive on the throttle for both drivers.
  - Lap 15 shows the same style as in C1, a long coast on the approach, light braking, and low throttle on the exit.

In [10]:
group_metrics("esses")

C2 apex at 37.1 m from window start
C3 apex at 49.1 m from window start
C4 apex at 59.1 m from window start
C5 apex at 69.1 m from window start


corner,2,3,4,5,2,3,4,5,2,3,4,5,2,3,4,5
lap,3,3,3,3,10,10,10,10,15,15,15,15,22,22,22,22
entry_from_apex,-35.60,-5.00,-8.40,-3.00,-35.30,-7.70,-7.50,-3.00,-10.10,-5.00,-6.30,-3.00,-34.70,-5.00,-5.70,-3.00
peak_brake_bar,11.48,0.34,0.00,0.00,16.08,4.87,0.04,0.04,3.77,4.51,0.00,0.00,9.55,0.00,0.01,0.00
brake_total_time,2.11,0.00,0.00,0.00,1.92,0.42,0.00,0.00,0.76,0.21,0.00,0.00,2.08,0.00,0.00,0.00
coasting_time,0.00,0.41,0.27,0.00,0.00,0.00,0.00,0.00,1.73,0.10,0.00,0.00,0.10,0.00,0.00,0.00
coasting_dist,0.00,2.40,2.70,0.00,0.00,0.00,0.00,0.00,24.00,0.90,0.00,0.00,2.40,0.00,0.00,0.00
min_speed,8.75,7.33,8.17,10.09,9.11,8.07,9.17,11.87,10.61,8.26,8.57,10.70,10.60,9.29,10.06,11.45
min_speed_from_apex,0.40,-2.30,-2.10,-2.80,-1.70,-2.90,-7.50,-2.80,0.40,-0.50,-1.80,-2.80,-0.50,-0.80,-5.70,-2.80
pickup_from_apex,NaN,-2.60,-3.00,NaN,-0.80,-3.20,NaN,NaN,NaN,-2.00,NaN,NaN,NaN,NaN,NaN,-2.20
max_exit_throttle,0.00,0.28,0.45,0.49,0.08,0.32,0.64,0.65,0.00,0.18,0.42,0.37,0.00,0.26,0.42,0.49


* **Metrics:**
  - Lap 22 brakes for about as long as lap 10 in C2 (2.08 s vs 1.92 s) but much lighter (9.55 bar vs 16.08 bar), and it comes out of C2 at 10.6 m/s against 9.1 m/s for lap 10. So the fastest lap is not braking less, it is braking lighter and carrying more speed into the corner.
  - Lap 10 is the only fast lap that brakes again for C3 (4.87 bar), lap 22 does not brake at all there. The minimum speed between C2 and C3 is 8.07 m/s for lap 10 against 9.29 m/s for lap 22, which shows how much speed lap 10 loses by slowing down too much in C2.
  - Lap 3 brakes the longest of all in C2 (2.11 s) and its minimum speed comes 0.4 m after the C2 apex, which matches the late brake release. It is also the slowest between C2 and C3 with a minimum speed of 7.33 m/s, and it coasts for 2.4 m in C3 and 2.7 m in C4, which is the coast before C4 that we see on the plot.
  - Lap 15 coasts for 24 m (1.73 s) before braking for C2, but its minimum speed in C2 (10.61 m/s) is the same as lap 22. So the coasting does not cost him speed at the apex, it costs him time on the approach, where he is much slower than everyone.
  - At the exit of the esses lap 10 is the fastest with 15.48 m/s, against 13.89 m/s for lap 22, 13.39 m/s for lap 3 and 12.54 m/s for lap 15.
  - Lap 10 has a very high rear wheel slip at the C5 exit (48%), and also 12.9% in C3. That is a lot of wheelspin, so either the driver is putting too much throttle on the exit, or this measurement is off for this lap (lap 10 also showed 42% at C24). This needs to be checked before we use it.
  - Corrections to the plot reading: lap 3 exits at 13.39 m/s, not 13.6 m/s. Lap 22 does not really coast between C2 and C3, it barely touches the throttle (around 0.02), so it is only 2.4 m of coasting in C2.